In [ ]:
!uv tree

# 0. Imports and Configuration Setup

In [ ]:
# ----- IMPORT -----
import os
import shutil # to remove database
import hashlib # create content's hash
import json
from pathlib import Path
from typing import List, Set, Dict, Any


# ----- LANGCHAIN IMPORTS -----
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document


# ----- FASTMCP IMPORT -----
# from fastmcp import FastMCP # direct import
from mcp.server.fastmcp import FastMCP

In [3]:
# ----- MCP SERVER INITIALIZATION -----
mcp = FastMCP("Research Assistant")


In [7]:
# ----- CONSTANTS -----
# current_dir = Path(__file__).parent.absolute() # not work in notebook
current_dir = Path.cwd()
current_dir

WindowsPath('c:/Users/user/Desktop/learning_mcp')

In [25]:
# ----- CONFIGURATION -----

CHROMA_DB_ROOT = os.path.join(current_dir, "research_chroma_dbs")
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"

# 1. Initialization

In [9]:
# ----- OLLAMA EMBEDDING INITIALIZATION -----
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

# 2. Utility Functions

We'll implement the following utility functions:

1. `def get_content_hash(content: str) -> str:` to get hash value of the content
2. `def load_content_hashes(topic_path: Path) -> Set[str]:` to load the (already hashed) stored content
3. `def save_content_hashes(topic_path: Path, hashes: Set[str]):` to store hashes
4. `def get_vectorstore(topic: str) -> Chroma:` to create vectorstore for a topic

### Experiment

In [14]:
text = "hello, 你好"
encoded_text = text.encode("utf-8")
encoded_text

b'hello, \xe4\xbd\xa0\xe5\xa5\xbd'

In [ ]:
hashlib.md5(encoded_text)

<md5 _hashlib.HASH object @ 0x0000024D67BF8430>

In [22]:
hashed_text = hashlib.md5(encoded_text).hexdigest()
print(f"hash length: {len(hashed_text)}")
hashed_text

hash length: 32


'67cb377af7afba7805f18f056feae8a0'

### 2.1 `get_content_hash()`

In [23]:
def get_content_hash(content: str) -> str:
    """Generate a hash for content to check for duplication."""
    return hashlib.md5(content.encode("utf-8")).hexdigest()

### 2.2 `load_content_hash()`

In [26]:
def load_content_hash(topic_path: Path) -> Set[str]:
    """Load existing content hashes from metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    if metadata_file.exists():
        try:
            with open(metadata_file, "r") as f:
                return set(json.load(f))
        except:
            return set()
    return set()

### 2.3 `save_content_hashes()`

In [ ]:
def save_content_hashes(topic_path: Path, hashes: Set[str]): # hashes created from get_content_hash()
    """Save content hashes to metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    with open(metadata_file, 'w') as f:
        json.dump(list(hashes), f)

### 2.4 `get_vectorstore()`

In [28]:
def get_vectorstore(topic: str) -> Chroma:
    """Get or create a ChromaDB vectorstore for a topic."""
    topic_path = CHROMA_DB_ROOT / topic
    topic_path.mkdir(parents=True, exist_ok=True)

    return Chroma(
        persist_directory=str(topic_path),
        embedding_function=embeddings,
        collection_name=f"research_on_{topic}"
    )

# 3. MCP Tools Implementations

In this section, we'll implement the following tools:

1. `def save_research_data(content: List[str], topic: str = "default") -> str:`
2. `def search_research_data(query: str, topic: str = "default", max_results: int = 5) -> str:`
3. `def list_research_topics() -> str:`
4. `def delete_research_topic(topic: str) -> str:`
5. `def get_topic_info(topic: str) -> str:`